# __1. Summary & Hypothesis__

- Act 1: The Tactical Approach (Local Baselines).
- __Goal:__ Establish that Gradient Boosting is superior for specific, high-frequency signals, and prove that Deep Learning fails without sufficient data scale.
- __Problem:__ Predicting "Foods in CA" (High volatility).
- __Hypothesis:__ A specialized Local Model should outperform a generic approach due to specific feature engineering.

# __2. Data Loading & Filtering__

### CONFIGURATION & DATA LOADING

In [1]:
import os
import torch
import pandas as pd
import numpy as np
import lightgbm as lgb
import gc

In [2]:
# def reduce_mem_usage(df, verbose=True):
def read_and_squeeze(file_name, INPUT_DIR_PATH = '../data/', verbose=True):
    
    # check if the file_name is a string
    if not isinstance(file_name, str):
        raise NotImplementedError(f'Read and squeeze function is not implemented yet. {file_name} must be a string.')
    # check if the file exists at the specified path
    if not os.path.exists(INPUT_DIR_PATH+file_name):
        raise NotImplementedError(f'Read and squeeze function is not implemented yet. "{file_name}" cannot be retrieved from "{INPUT_DIR_PATH}". Check if "INPUT_DIR_PATH" is correct.')

    df = pd.read_csv(INPUT_DIR_PATH+file_name)
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2    
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics: 
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)    
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose: 
        print(f'For {file_name}, mem. usage decreased from {start_mem:5.2f} Mb to {end_mem:5.2f} Mb ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    return df

In [3]:
# ==========================================
# 1. CONFIGURATION & DATA LOADING
# ==========================================
TARGET = 'sales'
START_DAY = 1000  # Skip the first few years to save memory/time
HORIZON = 28      # Forecast horizon

In [4]:
print("Loading Data...")
# Load only needed columns to save memory initially if possible, 
# but for simplicity we load and filter immediately.
df_sales = read_and_squeeze('sales_train_evaluation.csv')
df_cal = read_and_squeeze('calendar.csv')
df_prices = read_and_squeeze('sell_prices.csv')

Loading Data...
For sales_train_evaluation.csv, mem. usage decreased from 452.91 Mb to 96.13 Mb (78.8% reduction)
For calendar.csv, mem. usage decreased from  0.21 Mb to  0.12 Mb (41.9% reduction)
For sell_prices.csv, mem. usage decreased from 208.77 Mb to 130.48 Mb (37.5% reduction)


### FILTERING: FOODS & CA ONLY

- Training on a subset to allow rapid iteration.

In [5]:
# ==========================================
# 2. FILTERING: FOODS & CA ONLY
# ==========================================
print("Filtering for FOODS in CA...")
df_sales = df_sales[
    (df_sales['state_id'] == 'CA') & 
    (df_sales['cat_id'] == 'FOODS')
]

Filtering for FOODS in CA...


In [6]:
# ==========================================
# 3. MELTING (Wide to Long Format)
# ==========================================
# We need to turn the "d_1, d_2..." columns into rows
id_vars = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
df = pd.melt(df_sales, id_vars=id_vars, var_name='d', value_name='sales')

# Reduce memory usage
del df_sales
gc.collect()

0

In [7]:
display(df)

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3
1,FOODS_1_002_CA_1_evaluation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1,0
2,FOODS_1_003_CA_1_evaluation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,d_1,0
3,FOODS_1_004_CA_1_evaluation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA,d_1,0
4,FOODS_1_005_CA_1_evaluation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1,3
...,...,...,...,...,...,...,...,...
11156863,FOODS_3_823_CA_4_evaluation,FOODS_3_823,FOODS_3,FOODS,CA_4,CA,d_1941,0
11156864,FOODS_3_824_CA_4_evaluation,FOODS_3_824,FOODS_3,FOODS,CA_4,CA,d_1941,0
11156865,FOODS_3_825_CA_4_evaluation,FOODS_3_825,FOODS_3,FOODS,CA_4,CA,d_1941,0
11156866,FOODS_3_826_CA_4_evaluation,FOODS_3_826,FOODS_3,FOODS,CA_4,CA,d_1941,4


# __3. Feature Engineering (The "Tree" Way)__

### Create Lags (Lag-28 to Lag-49)

- This "Explicit Memory" is required for Trees, unlike RNNs

### Create Rolling Means

- This "Explicit Memory" is required for Trees, unlike RNNs

# __4. Model A: The Baseline (LightGBM)__

### Define the model

- Using `TweedieLoss` (choose $\rho=1.1$ for zero-inflation).

### Train the model

### Result

- Show RMSE (~2.6)

# __5. Model B: The Pivot (Local N-BEATSx)__

### Define N-BEATSx using `NeuralForecast`

- Use `HuberLoss` (explain the stability benefits over Tweedie for SGD).

### Train on the same subset

### Result

- Print RMSE (~2.84)

# __6. Conclusion: The "Data Hunger" Failure__

- Compare the two scores.
- __Verdict:__ _Deep Learning failed to beat the baseline. It converged to the mean because the local dataset lacks the diversity required for the model to learn generalizable temporal patterns. Next Step: Scale to the full dataset to enable Cross-Learning._